Usage:
------
The module expects three input parameters via Databricks widgets:


1. delivery_metadata (required):
    JSON structure containing delivery configuration:
    ```
    {
      "report_output": "/Volumes/d-cdh-delivery/temp/test/",
      "delivery_metadata": {
        "method": "s3",
        "s3_args": {
          "destination_path": "s3://d-ec1-cepal-common-zone/onprem-outgoing/test_output_reports/"
        }
      }
    }
    ```

2. reference_date (optional):
    - Format: YYYYMMDD
    - If not provided, **current date** is used

Example:
--------
```
{
  "report_output": "/Volumes/d-cdh-delivery/temp/test/",
  "delivery_metadata": {
    "method": "s3",
    "s3_args": {
      "destination_path": "s3://d-ec1-cepal-common-zone/onprem-outgoing/test_output_reports/"
    }
  }
}
```

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
    
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")

        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration

        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)    
else:
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

### INPUT PARAMETERS

```
"reference_date": "20241114"

"delivery_metadata":{
  "report_name": "Cetus_Monthly_Report",
  "report_output": "/Volumes/p-cdh-delivery/tia/tia",
  "delivery_args": {
    "method": "s3",
    "s3_args": {
      "destination_path": "s3://d-ec1-cepal-common-zone/onprem-outgoing/testfolder/"
    }
  }
}
```

In [0]:
dbutils.widgets.text("mode", "shared", "Mode")
dbutils.widgets.text("delivery_metadata", "", "Delivery Metadata")
dbutils.widgets.text("reference_date", "", "Reference Date")

In [0]:
try:
    delivery_metadata = dbutils.widgets.get("delivery_metadata")
except Exception as e:
    raise ValueError("Missing required input: delivery_metadata")

try:
    reference_date = dbutils.widgets.get("reference_date")
except Exception as e:
    reference_date = ""

In [0]:
import json

try:
    # Parse delivery_metadata
    delivery_metadata_json = json.loads(delivery_metadata)

    # Check delivery method. It must be "s3" (case insensitive)
    method = delivery_metadata_json.get('delivery_args', {}).get('method', '').lower()
    if method != 's3':
        raise Exception("This module is meant to deliver the reports on an S3 bucket but the delivery_args method tag is not 's3'. Please review the delivery job ID in your report if it meant to be delivered using some other module, or change the method to S3 and retry the call.")
    else:
        print(f"The delivery method is correct: {method}")

    # Check report_output is not blank and starts with "/volume"
    report_output = delivery_metadata_json.get('report_output', '')
    if not report_output or not report_output.startswith('/Volumes'):
        raise Exception("The report_output is either blank or does not start with '/Volumes'. Please ensure the report_output is correct and retry the call. This module supports transfering from Unity Catalog Volumes to S3.")
    else:
        print(f"The report_output is correct: {report_output}")

    # Check destination_path is an S3 bucket
    destination_path = delivery_metadata_json.get('delivery_args', {}).get('s3_args',{}).get('destination_path', '')
    if not destination_path.startswith('s3://'):
        raise Exception("The destination_path is not an S3 bucket. Please ensure the destination_path is correct and retry the call.")
    else:
        print(f"The destination_path is correct: {destination_path}")

except Exception as e:
    raise Exception(f"Error processing delivery metadata: {str(e)}")

In [0]:
from datetime import datetime
import os
from pyspark.dbutils import DBUtils
import boto3

# Initialize dbutils and S3 client
dbutils = DBUtils(spark)

# Define paths
#report_output = '/Volumes/d-cdh-delivery/temp/test/'
#destination_path = 's3://d-ec1-cepal-common-zone/onprem-outgoing/test_output_reports/'

# Determine the subfolder name
subfolder = reference_date if reference_date else datetime.now().strftime('%Y%m%d')

# Update paths with the subfolder
report_output = os.path.join(report_output, subfolder)
destination_path = os.path.join(destination_path, subfolder)

print(f"Moving files from {report_output} to {destination_path}")

# Extract bucket and prefix from destination path
bucket = destination_path.split('/')[2]
prefix = '/'.join(destination_path.split('/')[3:])

try:
    # List files in the Volume directory
    files_to_move = dbutils.fs.ls(report_output)
    
    # Create the subfolder in S3
    S3_client.put_object(Bucket=bucket, Key=prefix + '/')
    
    # Copy each file to S3
    for file in files_to_move:
        if not file.isDir():
            # Use dbutils to read the file content
            file_content = open(os.path.join(report_output, file.name), 'rb')
            
            # Construct the S3 key
            s3_key = os.path.join(prefix, file.name)
            
            # Upload to S3
            S3_client.put_object(
                Bucket=bucket,
                Key=s3_key,
                Body=file_content
            )
            
            print(f"Copied {file.name} to s3://{bucket}/{s3_key}")
    
    print("All files copied successfully")
except Exception as e:
    print(f"Error: {str(e)}")
